# FluxScript 0.1.0 — GitHub and PyPI Release Notebook

**Author:** Md Safarujjaman  
**Distribution:** `FluxScript`  
**Version:** `0.1.0`

This notebook prepares a clean release, runs tests, builds and validates the wheel/source archive, and optionally publishes to PyPI and pushes source to an existing GitHub repository.

**Security:** Tokens are requested through hidden prompts and are not written into source files. Never print, paste into a code cell, or commit a token. Prefer a project-scoped PyPI token or PyPI Trusted Publishing. The PyPI project name must be available to you. GitHub publishing requires an existing empty repository and a token with repository contents write access. Publishing is optional and requires explicit confirmation in the relevant cells.

Run top to bottom. Works best in Google Colab, but the upload cell also accepts a local file path in ordinary Jupyter.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile, tempfile, getpass, textwrap

WORK_ROOT = Path.cwd()
SOURCE_ZIP = None

def run(command, *, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

def locate_project(root: Path) -> Path:
    candidates = [root] + [p for p in root.rglob("pyproject.toml") if p.parent != root]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "fluxscript").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a FluxScript source folder containing pyproject.toml and fluxscript/.")

print("Python:", sys.version)
print("Working directory:", WORK_ROOT)

## 1. Upload the source ZIP

Upload `FluxScript-0.1.0-source.zip` from the project download. In Google Colab, a file picker opens. In local Jupyter, enter the ZIP path when prompted.

In [ ]:
try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No ZIP uploaded.")
    SOURCE_ZIP = Path(next(iter(uploaded.keys()))).resolve()
except ImportError:
    SOURCE_ZIP = Path(input("Path to FluxScript source ZIP: ").strip()).expanduser().resolve()

if not SOURCE_ZIP.is_file() or SOURCE_ZIP.suffix.lower() != ".zip":
    raise FileNotFoundError(f"Source ZIP not found or not a .zip file: {SOURCE_ZIP}")

EXTRACT_ROOT = WORK_ROOT / "fluxscript_release_input"
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    # Reject path traversal entries before extraction.
    for member in archive.infolist():
        destination = (EXTRACT_ROOT / member.filename).resolve()
        if destination != EXTRACT_ROOT.resolve() and EXTRACT_ROOT.resolve() not in destination.parents:
            raise ValueError(f"Unsafe ZIP path: {member.filename}")
    archive.extractall(EXTRACT_ROOT)
PROJECT = locate_project(EXTRACT_ROOT)
print("Source project:", PROJECT)
print((PROJECT / "pyproject.toml").read_text(encoding="utf-8"))

## 2. Install release tools and verify metadata

This installs the packaging tools in the notebook runtime only. It does not modify the FluxScript runtime dependency list.

In [ ]:
run([sys.executable, "-m", "pip", "install", "--upgrade", "build", "twine"])

try:
    import tomllib
except ImportError:  # Python 3.10
    run([sys.executable, "-m", "pip", "install", "tomli"])
    import tomli as tomllib
metadata = tomllib.loads((PROJECT / "pyproject.toml").read_text(encoding="utf-8"))
project_meta = metadata["project"]
assert project_meta["name"].lower() == "fluxscript", project_meta["name"]
assert project_meta["version"] == "0.1.0", project_meta["version"]
assert project_meta.get("authors", [{}])[0].get("name") == "Md Safarujjaman"
print("Metadata checks passed:", project_meta["name"], project_meta["version"], project_meta["authors"])

## 3. Run tests and build clean distributions

The source ZIP may contain previously generated artifacts; this step removes them and builds fresh artifacts from the checked source.

In [ ]:
dist_dir = PROJECT / "dist"
if dist_dir.exists():
    shutil.rmtree(dist_dir)
for generated in PROJECT.glob("*.egg-info"):
    shutil.rmtree(generated, ignore_errors=True)

run([sys.executable, "-m", "compileall", "-q", "fluxscript", "tests"], cwd=PROJECT)
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT)
run([sys.executable, "-m", "build"], cwd=PROJECT)
run([sys.executable, "-m", "twine", "check", *sorted(dist_dir.glob("*"))], cwd=PROJECT)
print("Release artifacts:")
for artifact in sorted(dist_dir.glob("*")):
    print(artifact.name, artifact.stat().st_size, "bytes")

## 4. Optional: publish to PyPI

Confirm that the `FluxScript` project name is available and that you control it before continuing. For an existing project, use a project-scoped API token. The token is read with a hidden prompt and passed to Twine only through environment variables.

In [ ]:
publish_pypi = input("Publish these artifacts to PyPI now? Type YES to confirm: ").strip()
if publish_pypi == "YES":
    pypi_token = getpass.getpass("PyPI API token (input hidden): ").strip()
    if not pypi_token.startswith("pypi-"):
        raise ValueError("This does not look like a PyPI API token. Nothing was uploaded.")
    env = os.environ.copy()
    env["TWINE_USERNAME"] = "__token__"
    env["TWINE_PASSWORD"] = pypi_token
    env["TWINE_NON_INTERACTIVE"] = "1"
    try:
        run([sys.executable, "-m", "twine", "upload", "--non-interactive", *sorted(dist_dir.glob("*"))], cwd=PROJECT, env=env)
        print("PyPI upload command completed. Verify the release on your PyPI project page.")
    finally:
        env.pop("TWINE_PASSWORD", None)
        pypi_token = None
else:
    print("Skipped PyPI upload. Artifacts remain in:", dist_dir)

## 5. Optional: push source to GitHub

Create an **empty** GitHub repository first. This cell pushes the source tree to its `main` branch. A token is entered with a hidden prompt and is not embedded in the remote URL or saved in the repository. Do not use a personal token with broader permissions than necessary.

In [ ]:
push_github = input("Push this source tree to an existing GitHub repository? Type YES to confirm: ").strip()
if push_github == "YES":
    repo_url = input("HTTPS URL of your empty GitHub repository (for example https://github.com/USER/REPO.git): ").strip()
    if not repo_url.startswith("https://github.com/") or not repo_url.endswith(".git"):
        raise ValueError("Use an HTTPS GitHub repository URL ending in .git. No push was attempted.")
    github_token = getpass.getpass("GitHub token (hidden; needs repository contents write access): ").strip()
    if not github_token:
        raise ValueError("GitHub token is empty.")

    run(["git", "--version"], cwd=PROJECT)
    run(["git", "init", "-b", "main"], cwd=PROJECT)
    run(["git", "config", "user.name", "Md Safarujjaman"], cwd=PROJECT)
    commit_email = input("Git commit email (public email or GitHub noreply email): ").strip()
    if not commit_email or "\n" in commit_email:
        raise ValueError("A valid one-line commit email is required.")
    run(["git", "config", "user.email", commit_email], cwd=PROJECT)
    run(["git", "add", "-A"], cwd=PROJECT)
    run(["git", "commit", "-m", "Release FluxScript 0.1.0"], cwd=PROJECT)
    # Configure askpass in a temporary file; the secret is supplied only via env.
    askpass_dir = Path(tempfile.mkdtemp(prefix="fluxscript-askpass-"))
    askpass = askpass_dir / "askpass.py"
    askpass.write_text(
        "import os, sys\nprompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
        "print('x-access-token' if 'username' in prompt else os.environ.get('FLUXSCRIPT_GITHUB_TOKEN', ''))\n",
        encoding="utf-8",
    )
    env = os.environ.copy()
    env["GIT_ASKPASS"] = sys.executable + " " + str(askpass)
    # GIT_ASKPASS is normally an executable path; use a temporary executable wrapper.
    wrapper = askpass_dir / "askpass"
    wrapper.write_text("#!/bin/sh\nexec "+repr(sys.executable)+" "+repr(str(askpass))+" \"$@\"\n", encoding="utf-8")
    wrapper.chmod(0o700)
    env["GIT_ASKPASS"] = str(wrapper)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["FLUXSCRIPT_GITHUB_TOKEN"] = github_token
    try:
        remotes = subprocess.run(["git", "remote"], cwd=PROJECT, capture_output=True, text=True, check=True).stdout.split()
        if "origin" in remotes:
            run(["git", "remote", "set-url", "origin", repo_url], cwd=PROJECT)
        else:
            run(["git", "remote", "add", "origin", repo_url], cwd=PROJECT)
        run(["git", "push", "-u", "origin", "main"], cwd=PROJECT, env=env)
        print("GitHub push completed. Verify the repository page.")
    finally:
        env.pop("FLUXSCRIPT_GITHUB_TOKEN", None)
        github_token = None
        shutil.rmtree(askpass_dir, ignore_errors=True)
else:
    print("Skipped GitHub push. Source remains in:", PROJECT)

## 6. Final checklist

- [ ] Confirm the PyPI project name `FluxScript` is available or belongs to you.
- [ ] Check package metadata, license, README and release notes.
- [ ] Review the test output and `twine check` results.
- [ ] Prefer TestPyPI or a protected GitHub Actions Trusted Publishing workflow before a production release.
- [ ] Verify the PyPI release and GitHub repository after upload.

**This notebook does not publish anything unless you explicitly type `YES` in the corresponding cell.** Version stays `0.1.0`; PyPI does not allow replacing an already-published file with the same version, so any changes after publishing require a new version.